# 1. Context

This notebook assesses reduction in error rates in samples where CER or WER >= 10%. This helps in identifying Model & Prompt Combination which could help in Post OCR Correction

# 2. Imports

In [38]:
import pandas as pd
from pathlib import Path
from collections import defaultdict
import sys
import tqdm
import torch

In [39]:
from transformers import AutoTokenizer, AutoModelForCausalLM

In [40]:
import jiwer

In [41]:
from indicnlp.normalize.indic_normalize import IndicNormalizerFactory

In [42]:
notebook_path = Path()
sys.path.append(str(notebook_path.resolve().parent.parent))

In [43]:
from src.evaluation.metrics import cer, wer
from src.common.constants import language_code_norm_map, language_to_writing_system
from src.common.utils import get_script_results

# 3. Utils

## 4.1. Loading Results

In [44]:
# get csv paths for all language
results_root = Path("../../results/surya_ocr")
results_langs = [x.name.lower() for x in results_root.glob("*") if x.is_dir()]

In [45]:
writing_sys_dict = defaultdict(list)
for language_res in results_langs:
    script = language_to_writing_system.get(language_res)[0]
    writing_sys_dict[script].append(language_res)

script_language_result = pd.Series(writing_sys_dict).to_frame(name='languages')
script_language_result.index.name = 'script'

In [46]:
script_results_avail = script_language_result.index
results_consolidated = []
for script in script_results_avail:
    results_script = get_script_results(script=script, script_lang_df=script_language_result, result_root=results_root)
    results_consolidated.append(results_script)
consolidated_df = pd.concat(results_consolidated)

# 4. Computing Err Rates

## 4.1. UTC Norm

In [47]:
def normalize_text(text: str, lang: str, norm_factory: IndicNormalizerFactory):
    """Normalize text for a given language using indic normalizer"""
    if lang not in language_code_norm_map:
        return text
    code = language_code_norm_map[lang]
    normalizer = norm_factory.get_normalizer(code)
    normalized_text = normalizer.normalize(text)
    return normalized_text

In [48]:
norm_factory = IndicNormalizerFactory()

In [49]:
cols_to_norm = ['ground_truth', 'ocr_output_L_0', 'ocr_output_L_1',
       'ocr_output_L_2', 'ocr_output_L_3']
for col in cols_to_norm:
    consolidated_df[col] = consolidated_df.apply(lambda x: normalize_text(x[col], x['language'], norm_factory), axis=1)

## 4.2. Error Estimates

In [50]:
# computing cer & wer
gt_col = 'ground_truth'
ocr_output_cols = ['ocr_output_L_0', 'ocr_output_L_1', 'ocr_output_L_2', 'ocr_output_L_3']
cols_create_cer = ['cer_l0', 'cer_l1', 'cer_l2', 'cer_l3']
cols_create_wer = ['wer_l0', 'wer_l1', 'wer_l2', 'wer_l3']
for ocr_output_lvl, col_crt_cer in dict(zip(ocr_output_cols, cols_create_cer)).items():
    consolidated_df[col_crt_cer] = consolidated_df[[gt_col, ocr_output_lvl]].apply(lambda x: cer(x[gt_col], x[ocr_output_lvl]), axis=1)
for ocr_output_lvl, col_crt_wer in dict(zip(ocr_output_cols, cols_create_wer)).items():
    consolidated_df[col_crt_wer] = consolidated_df[[gt_col, ocr_output_lvl]].apply(lambda x: wer(x[gt_col], x[ocr_output_lvl]), axis=1)

In [51]:
col_ord = ['file_id', 'language', 'script','ground_truth', 'ocr_output_L_0', 'ocr_output_L_1',
       'ocr_output_L_2', 'ocr_output_L_3', 'cer_l0', 'cer_l1', 'cer_l2',
       'cer_l3', 'wer_l0', 'wer_l1', 'wer_l2', 'wer_l3' ]
consolidated_df = consolidated_df[col_ord]

## upper casing column names
consolidated_df.columns = consolidated_df.columns.str.upper()

# 5. Selecting Text Blocks For Analysis

In [52]:
_LANGUGAE_EXCLUDE_ = ["santali", "manipuri"]
results_lang_scoped = consolidated_df[~consolidated_df['LANGUAGE'].isin(_LANGUGAE_EXCLUDE_)]

In [53]:
# degradation level & ocr output col map
degradation_ocr_col_map = {
    'L0': 'ocr_output_L_0',
    'L1': 'ocr_output_L_1',
    'L2': 'ocr_output_L_2',
    'L3': 'ocr_output_L_3'
}

In [54]:
filtered_rows_list = []
common_cols = ['FILE_ID', 'SCRIPT','LANGUAGE', 'GROUND_TRUTH', ]
WER_COLS = ['WER_L0', 'WER_L1', 'WER_L2', 'WER_L3']
CER_COLS = ['CER_L0', 'CER_L1', 'CER_L2', 'CER_L3']

for idx, row in results_lang_scoped.iterrows():
    selected_rows_base = row[common_cols]
    for CER in CER_COLS:
        if row[CER] >= 0.1:
            degradation_id = CER.split('_')[-1]
            ocr_output_col = degradation_ocr_col_map[degradation_id].upper()
            selected_values = row[common_cols + [ocr_output_col] ]
            selected_values.rename({ocr_output_col: 'OCR_OUTPUT'}, inplace=True)
            selected_values['DEGRADTION_LEVEL'] = degradation_id
            filtered_rows_list.append(selected_values)
            break
    for WER in WER_COLS:
        if row[WER] >= 0.1:
            degradation_id = WER.split('_')[-1]
            ocr_output_col = degradation_ocr_col_map[degradation_id].upper()
            selected_values = row[common_cols + [ocr_output_col] ]
            selected_values.rename({ocr_output_col: 'OCR_OUTPUT',}, inplace=True)
            selected_values['DEGRADTION_LEVEL'] = degradation_id
            filtered_rows_list.append(selected_values)
    filtered_rows_list.append(selected_values)
    

In [55]:
results_high_err = pd.DataFrame(filtered_rows_list).drop_duplicates()

# 6. Prompt & Model Experiments

In [56]:
sample_results_high_err = results_high_err.loc[results_high_err['LANGUAGE'] == 'hindi']

In [57]:
GT = sample_results_high_err.iloc[7]['GROUND_TRUTH']
OCR = sample_results_high_err.iloc[7]['OCR_OUTPUT']

In [58]:
print(f"CER: {jiwer.cer(GT, OCR)}")

CER: 0.12411575562700965


In [59]:
print(f"CER: {jiwer.wer(GT, OCR)}")

CER: 0.14469453376205788


In [60]:
torch.device("cuda")

device(type='cuda')

## 6.1. WIP Model Loader Class

In [62]:
class postCorrectorLLM():
    """Class For OCR Post Correction Using a LLM"""

    @staticmethod
    def _infer_device():
        if torch.cuda.is_available():
            device = torch.device("cuda")
        # Check for MPS (Apple Silicon Mac)
        elif torch.backends.mps.is_available():
            device = torch.device("mps")
        else:
            device = torch.device("cpu")
        return device
    
    def __init__(self, model_id: str):
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        self.device = self._infer_device()
        self.model = AutoModelForCausalLM.from_pretrained(model_id, device_map=self.device)
        self.model_id = model_id
    def _prepare_input_prompt(self, context: str):
        """Converts context to input token ids"""
        messages = [{"role": "user", "content": context}]
        input_tokens = self.tokenizer.apply_chat_template(messages, 
                                                          add_generation_prompt=True, 
                                                          tokenize=True, 
                                                          return_dict=True, 
                                                          return_tensors="pt").to(self.model.device)
        return input_tokens

    def __call__(self, context: str, **kwargs):
        input_tokens = self._prepare_input_prompt(context=context)
        model_outputs = self.model.generate(**input_tokens, max_new_tokens=1000,**kwargs)
        decoded_resp = self.tokenizer.decode(model_outputs[0][input_tokens["input_ids"].shape[-1]:], skip_special_tokens=True)
        return decoded_resp

In [63]:
llm_obj = postCorrectorLLM(model_id="google/gemma-3-1b-it")

In [26]:
OCR_CORRECTOR_PROMPT_TEMPLATE_1 = """You are a professional text proof reader. 
You will be provided with text to be corrected in {text} placeholder.
Strictly Follow Instruction Below
- Preserve original text length
- Rectify any character or word error present and provide corrected output in {corrected_text}
Directly provide corrected text without any additional information.
text: {ocr_output}
"""

In [27]:
OCR_CORRECTOR_PROMPT_TEMPLATE_2 = """You are an expert OCR correction agent. Your task is to correct transcription errors in the provided text, which may be in any language.

You must first implicitly identify the language of the text to ensure your corrections are accurate. Then, analyze the text for common OCR errors, including:
- Character misrecognition (e.g., '1' for 'l', 'O' for '0', 'rn' for 'm').
- Incorrectly merged or split words.
- Punctuation and diacritical mark mistakes (e.g., accents, umlauts).

Correct these errors while strictly preserving the original formatting, such as line breaks, indentation, and overall structure. Make only the minimum changes necessary.

Do not add any commentary, explanations, or introductory phrases. Output only the corrected text.

--- OCR TEXT ---
{ocr_output}"""

In [64]:
CONTEXT = OCR_CORRECTOR_PROMPT_TEMPLATE_2.replace("{ocr_output}", OCR)

In [65]:
llm_obj(CONTEXT)

"की भूख पक्ष एकादशी तब हिंदू धर्म में एकादशी का व्रत महत्वपूर्ण स्थान रखता है। प्रत्येक वर्ष चौबीस एकादशियाँ होती हैं। जब अधिकमास या मलमास आता है तब इनकी संख्या बढ़कर २६ हो जाती है। ज्येष्ठ मास की शुक्ल पक्ष की जावकुनाल वा नुलनाल जाता हु तब इनका लख्या अकुकर रुव हा जाता हा ज्यूछ माल का सुक्ला पक्ष का एकादशी को निर्जला एकादशी कहते है इस व्रत में पानी का पीना वर्जित है इसिलिये इस निर्जला एकादशी कहते है। जब सर्वज्ञ वेदव्यास ने पांडवों को चारों पुरुषार्थ- धर्म, अर्थ, काम और मोक्ष देने वाले एकादशी व्रत का संकल्प कराया तो महाबली भीम ने निवेदन किया- पितामह! आपने तो प्रति पक्ष एक दिन के उपवास की बात कही है। मैं तो एक दिन क्या एक समय भी भोजन के बगैर नहीं रह सकता- मेरे पेट में 'वृक' नाम की जो अग्नि है, उसे शांत रखने के लिए मुझे कई लोगों के बराबर और कई बार भोजन करना पड़ता है। तो क्या अपनी उस भूख के कारण मैं एकादशी जैसे पुण्यव्रत से वंचित रह जाऊँगा? पितामह ने भीम की समस्या का निदान करते और उनका मनोबल बढ़ाते हुए कहा- नहीं कुंतीनंदन, धर्म की यही तो विशेषता है कि वह सबको धारण ही नहीं करता, सबके योग्य 

In [66]:
sample_results_high_err

,FILE_ID,SCRIPT,LANGUAGE,GROUND_TRUTH,OCR_OUTPUT,DEGRADTION_LEVEL
0,Devanagari_Anek_Devanagari_17,Devanagari,hindi,यह में स्नातक यहाँ में इन्द्रप्रस्थ महिला महाव...,यह में स्नातक यहाँ में में हुई थी। यह दिल्ली व...,L3
0,Devanagari_Anek_Devanagari_17,Devanagari,hindi,यह में स्नातक यहाँ में इन्द्रप्रस्थ महिला महाव...,यह में स्नातक यहाँ में इन्द्रप्रस्थ महिला महाव...,L1
1,Devanagari_Poppins_15,Devanagari,hindi,उन्नत स्वयं एडोब एकाधिक वास्तविक अडोबी फोटोशॉप...,उन्नत स्वयं एडोब एकाधिक वास्तविक अड़ोबी फोटोशॉ...,L2
1,Devanagari_Poppins_15,Devanagari,hindi,उन्नत स्वयं एडोब एकाधिक वास्तविक अडोबी फोटोशॉप...,उन्नत स्वयं एडोब एकाधिक वास्तविक अड़ोबी फोटोशॉ...,L3
2,Devanagari_Noto_Sans_10,Devanagari,hindi,1970 आम यह पट्टे में लोकल एरिया नेटवर्क (LAN) ...,1970 आम यह पट्टे में लोकल एरिया नेटवर्क (LAN) ...,L3
5,Devanagari_Noto_Serif_Devanagari_16,Devanagari,hindi,की भूख पक्ष एकादशी तब हिंदू धर्म में एकादशी का...,की भूख पक्ष एकादशी तब हिंदू धर्म में एकादशी का...,L0
5,Devanagari_Noto_Serif_Devanagari_16,Devanagari,hindi,की भूख पक्ष एकादशी तब हिंदू धर्म में एकादशी का...,की भूख पक्ष एकादशी तब हिंदू धर्म में एकादशी का...,L1
5,Devanagari_Noto_Serif_Devanagari_16,Devanagari,hindi,की भूख पक्ष एकादशी तब हिंदू धर्म में एकादशी का...,की भूख पक्ष एकादशी तब हिंदू धर्म में एकादशी का...,L2
6,Devanagari_Anek_Devanagari_26,Devanagari,hindi,है। समस्थानिकों (प्रोटियम). है। एकमात्र हाइड्र...,है। समस्थानिकों (प्रोटियम). है। एकमात्र हाइड्र...,L3
6,Devanagari_Anek_Devanagari_26,Devanagari,hindi,है। समस्थानिकों (प्रोटियम). है। एकमात्र हाइड्र...,है। समस्थानिकों (प्रोटियम). है। एकमात्र हाइड्र...,L0


In [33]:
GenerationConfig = {
    "top_p": 0.9,
    "temperature": 0.0,
    "do_sample": False
}

resp_list = []
for idx, row in sample_results_high_err.head(10).reset_index().iterrows():
    print(idx)
    OCR_OUTPUT = row['OCR_OUTPUT']
    CONTEXT = OCR_CORRECTOR_PROMPT_TEMPLATE_2.replace("{ocr_output}", OCR_OUTPUT)
    resp_list.append(llm_obj(CONTEXT , kwargs=GenerationConfig))

0
1
2
3
4
5
6
7
8
9


In [35]:
resp_list

['यह में स्नातक यहाँ में हुई थी। यह दिल्ली विश्वविद्यालय जिसे इंद्रप्रस्थ कॉलेज या आईपी कॉलेज भी कहा जाता है, की स्थापना १९२४ में छिप्पीवाड़े में एक पुरानी हवेली की दूसरी मंजिल में स्थित कमरे में तीन छात्राओं से हुआ। १९२० में छिप्पीवाड़े में एक पुरानी हवेली की दूसरी मंजिल में स्थित कमरे में तीन छात्राओं से हुआ। १९३० में स्नातक पाठ्यक्रम आरंभ हुए एवं १९३८ में विश्वविद्यालय द्वारा इन्द्रप्रस्थ महाविद्यालय को सबाये पे पुराना महिला महाविद्यालय के अर्थ महावाद्यालय के जामा रूप में मान्यता मिली। कुछ वर्ष पश्चात् यह महाविद्यालय सिविल लाइन्स क्षेत्र के चन्द्रावली मतक महाविद्यालय के दिया गया और तदुपरांत १९३८ में इसे ब्रिटिश कमांडर-इन-चीफ के अलीपुर रोड (वर्तमान शाम नाथ मार्ग) स्थित अलीपुर हाउस वाले कार्यालय-सह-आवास में पुनः स्थानांतरित कर दिया गया। थियोसॉफिकल सोसाइटी ऑफ इंडिया से संबद्ध समाजसेवियों के प्रयासों से मूल विद्यालय और महाविद्यालय विकसित हुआ। उन्हें थियोसॉफिस्ट श्रीमती एनी बेसेंट् से प्रुरणा मिली थी। एनी बेसेंट ने उत्तर भारत की महिलाओं को शिक्षित करने का उस समय बीड्रा उठाया जबकि महिलाएँ